## C-Drama RAG: Data Exploration
**Goals:** 
- Understand the shape of the data
- Find nulls, weird values and decide what needs cleaning before inserting into Supabase

Important fields for embedding:
- **title**, **synopsis**, **genres**, **tags**

### Setup & Data Loading
data saved as NDJSON - newline-delimited JSON, each line is one drama

In [36]:
import polars as pl
from pathlib import Path


DATA_PATH = Path("../data/raw/dramas.ndjson")
df = pl.read_ndjson(DATA_PATH)

### Check quality of data, identify scraping errors before cleaning
focus is on dramas with:
- rating of 7.5 and higher 
- 10 or more episodes (to filter out specials and extras)
- 500 or more watchers (to get good dramas from the recommender)

#### 1. Shape and schema

In [17]:
print(f"Shape: {df.shape}")  
df.head(3)

Shape: (2694, 16)


mdl_id,mdl_url,title,native_title,also_known_as,synopsis,country,episodes,duration_min,content_rating,network,year,genres,tags,mdl_score,watchers
i64,str,str,str,str,str,str,i64,i64,str,str,i64,list[str],list[str],f64,i64
53505,"""https://mydramalist.com/53505-…","""The Untamed Special Edition""","""陈情令特別剪辑版""","""Os Indomáveis Edição Especial …","""A 20-episode cut of The Untame…","""China""",20,1,"""15+ - Teens 15 or older""","""Tencent Video""",2020,"[""Mystery"", ""Wuxia"", ""Fantasy""]","[""Bromance"", ""Smart Male Lead"", … ""Cultivation""]",9.0,38663
9025,"""https://mydramalist.com/9025-n…","""Nirvana in Fire""","""琅琊榜""","""Lang Ya Bang , Nirvana in Fir…","""In sixth-century China, the Em…","""China""",54,45,"""13+ - Teens 13 or older""","""BTV""",2015,"[""Military"", ""Historical"", … ""Political""]","[""Power Struggle"", ""Smart Male Lead"", … ""Sibling Rivalry""]",9.0,59925
62295,"""https://mydramalist.com/62295-…","""When I Fly Towards You""","""当我飞奔向你""","""The Girl Who Fell Into Love , …","""In the autumn of 2012, Su Zai …","""China""",24,35,"""13+ - Teens 13 or older""","""Youku""",2023,"[""Comedy"", ""Romance"", ""Youth""]","[""Friendship"", ""Love At First Sight"", … ""Introverted Male Lead""]",9.0,97911


In [18]:
# important to check if Polars inferred data types correctly
df.schema

Schema([('mdl_id', Int64),
        ('mdl_url', String),
        ('title', String),
        ('native_title', String),
        ('also_known_as', String),
        ('synopsis', String),
        ('country', String),
        ('episodes', Int64),
        ('duration_min', Int64),
        ('content_rating', String),
        ('network', String),
        ('year', Int64),
        ('genres', List(String)),
        ('tags', List(String)),
        ('mdl_score', Float64),
        ('watchers', Int64)])

In [8]:
# basic stats on numeric columns
df.select(["mdl_score", "watchers", "episodes", "duration_min"]).describe()

statistic,mdl_score,watchers,episodes,duration_min
str,f64,f64,f64,f64
"""count""",2693.0,2694.0,2694.0,2645.0
"""null_count""",1.0,0.0,0.0,49.0
"""mean""",9.3329,5220.199332,34.06199,37.16862
"""std""",16.862435,10856.088931,19.529716,14.422714
"""min""",2.0,2.0,1.0,1.0
"""25%""",7.6,100.0,24.0,35.0
"""50%""",7.8,836.0,32.0,45.0
"""75%""",8.1,5537.0,40.0,45.0
"""max""",548.0,147271.0,367.0,60.0


#### 2. Score distribution

In [20]:
score_dist = (
    df.group_by(
        pl.col("mdl_score")
    )
    .agg(pl.len().alias("count"))
    .sort("mdl_score")
)
print(score_dist)

# --- Sanity checks ---
print(f"\nDramas below 7.5: {df.filter(pl.col('mdl_score') < 7.5).height}")

# Dramas rated above 9 — likely real but voted by very few people, unreliable
above_9 = df.filter(pl.col("mdl_score") > 9.0)
print(f"\nDramas rated above 9.0: {above_9.height}")
print(above_9.select(["title", "mdl_score", "watchers"]).sort("mdl_score", descending=True))

# Scraper bug — impossibly high scores
print(f"\nDramas with score > 10 (scraper bug): {df.filter(pl.col('mdl_score') > 10).height}")
print(df.filter(pl.col('mdl_score') > 10).select(["title", "mdl_score", "watchers"]))

shape: (73, 2)
┌───────────┬───────┐
│ mdl_score ┆ count │
│ ---       ┆ ---   │
│ f64       ┆ u32   │
╞═══════════╪═══════╡
│ null      ┆ 1     │
│ 2.0       ┆ 1     │
│ 4.0       ┆ 1     │
│ 6.0       ┆ 4     │
│ 7.0       ┆ 2     │
│ …         ┆ …     │
│ 143.0     ┆ 1     │
│ 144.0     ┆ 1     │
│ 197.0     ┆ 1     │
│ 517.0     ┆ 1     │
│ 548.0     ┆ 1     │
└───────────┴───────┘

Dramas below 7.5: 11

Dramas rated above 9.0: 192
shape: (192, 3)
┌─────────────────────────────────┬───────────┬──────────┐
│ title                           ┆ mdl_score ┆ watchers │
│ ---                             ┆ ---       ┆ ---      │
│ str                             ┆ f64       ┆ i64      │
╞═════════════════════════════════╪═══════════╪══════════╡
│ Gorgeous Workers                ┆ 548.0     ┆ 548      │
│ Secret Society of Men - Friend… ┆ 517.0     ┆ 517      │
│ Young Justice Bao               ┆ 197.0     ┆ 197      │
│ Young Justice Bao Season 3      ┆ 144.0     ┆ 144      │
│ Xiao Zhuang

When a drama has an "N/A" score, the scraper accidentally grabs the "watcher" count instead (resulting in impossible scores like 548.0).
Dramas with very few watchers (e.g., < 100) often have a perfect 10.0 score because they lack a true community consensus.

#### 3. Watchers distribution

In [35]:
watcher_buckets = (
    df.with_columns(
        pl.when(pl.col("watchers") < 500).then(pl.lit("< 500"))
        .when(pl.col("watchers") < 5_000).then(pl.lit("1k–5k"))
        .when(pl.col("watchers") < 20_000).then(pl.lit("5k–20k"))
        .when(pl.col("watchers") < 100_000).then(pl.lit("20k–100k"))
        .otherwise(pl.lit("100k+"))
        .alias("watcher_bucket")
    )
    .group_by("watcher_bucket")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print(watcher_buckets)

shape: (5, 2)
┌────────────────┬───────┐
│ watcher_bucket ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ < 500          ┆ 1176  │
│ 1k–5k          ┆ 807   │
│ 5k–20k         ┆ 527   │
│ 20k–100k       ┆ 179   │
│ 100k+          ┆ 5     │
└────────────────┴───────┘


#### 4. Episodes distribution

In [33]:
episode_dist = (
    df.group_by("episodes")
    .agg(pl.len().alias("count"))
    .sort("episodes")
)

print("--- Very short (<10 episodes) ---")
print(episode_dist.filter(pl.col("episodes") < 10))


--- Very short (<10 episodes) ---
shape: (9, 2)
┌──────────┬───────┐
│ episodes ┆ count │
│ ---      ┆ ---   │
│ i64      ┆ u32   │
╞══════════╪═══════╡
│ 1        ┆ 57    │
│ 2        ┆ 7     │
│ 3        ┆ 7     │
│ 4        ┆ 14    │
│ 5        ┆ 8     │
│ 6        ┆ 13    │
│ 7        ┆ 7     │
│ 8        ┆ 17    │
│ 9        ┆ 2     │
└──────────┴───────┘


#### 5. Null values

In [26]:
df.null_count()



mdl_id,mdl_url,title,native_title,also_known_as,synopsis,country,episodes,duration_min,content_rating,network,year,genres,tags,mdl_score,watchers
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,1,0,0,0,0,49,0,537,82,0,0,1,0


In [27]:
# Check for empty strings in synopsis
empty_synopsis = df.filter(
    pl.col("synopsis").is_null() | (pl.col("synopsis").str.strip_chars() == "")
)
print(f"Dramas with missing or empty synopsis: {empty_synopsis.height}")

Dramas with missing or empty synopsis: 0


#### 6. Duplicate check
to make sure there isn't same drama scraped twice

In [29]:
duplicates = df.filter(pl.col("mdl_id").is_duplicated())
print(f"Duplicate mdl_ids: {df.filter(pl.col('mdl_id').is_duplicated()).height}")
print(f"Fully duplicate rows: {df.height - df.unique().height}")

Duplicate mdl_ids: 0
Fully duplicate rows: 0


#### 7. Synopsis quality check

In [30]:
# quick look at synopsis
df["synopsis"][1]

'In sixth-century China, the Emperor of Great Liang orders the unjust execution of his brother-in-law Marshal Lin Xie alongside the Lin family, his 70,000 army soldiers, and Crown Prince Qi. Secretly surviving the massacre is Lin Xie\'s son, Lin Shu, who undergoes medical treatment that changes his appearance entirely and leaves him in a weakened state, unable to ever perform martial arts again. Lin Shu changed his name to Mei Chang Suand later became the chief of the pugilist world and established the Jiangzuo Alliance.\n\nTwelve years later, Mei Chang Su returns to the capital with a secret plan after being sought after by Prince Yu and Prince Xian during their fight for the throne. He decides to covertly assist Prince Jing, the unfavoured son of the Emperor, and wisely rids the court of all scheming officials.\n\n(Source: MyDramaList)\n\n~~ Adapted from the novel "Lang Ya Bang" (琅琊榜) by Hai Yan (海宴).Edit TranslationEnglish한국어中文(简体)ภาษาไทย'

Most dramas has \n characters, (Source), Adapted from, Translation
`\n\n(Source: MyDramaList)\n\n~~ Adapted from the novel "Lang Ya Bang" (琅琊榜) by Hai Yan (海宴).Edit TranslationEnglish한국어中文(简体)ภาษาไทย'`

In [31]:
synopsis_stats = df.select([
    pl.col("synopsis").str.len_chars().mean().alias("avg_length"),
    pl.col("synopsis").str.len_chars().min().alias("min_length"),
    pl.col("synopsis").str.len_chars().max().alias("max_length"),
])
print(synopsis_stats)

# Check for the "Edit Translation" junk we saw in your sample
junk_count = df.filter(
    pl.col("synopsis").str.contains("Edit Translation")
).height
print(f"\nSynopses with 'Edit Translation' junk: {junk_count}")

# Peek at very short synopses 
print("\n--- Shortest synopses ---")
print(
    df.filter(pl.col("synopsis").is_not_null())
    .select(["title", "synopsis"])
    .with_columns(pl.col("synopsis").str.len_chars().alias("length"))
    .sort("length")
    .head(5)
)

shape: (1, 3)
┌────────────┬────────────┬────────────┐
│ avg_length ┆ min_length ┆ max_length │
│ ---        ┆ ---        ┆ ---        │
│ f64        ┆ u32        ┆ u32        │
╞════════════╪════════════╪════════════╡
│ 591.604677 ┆ 50         ┆ 5749       │
└────────────┴────────────┴────────────┘

Synopses with 'Edit Translation' junk: 2694

--- Shortest synopses ---
shape: (5, 3)
┌────────────────────────────────┬─────────────────────────────────┬────────┐
│ title                          ┆ synopsis                        ┆ length │
│ ---                            ┆ ---                             ┆ ---    │
│ str                            ┆ str                             ┆ u32    │
╞════════════════════════════════╪═════════════════════════════════╪════════╡
│ Five Star Hotel                ┆ Edit TranslationEnglishPortugu… ┆ 50     │
│ Only for Love Special          ┆ Edit TranslationEnglishEspañol… ┆ 51     │
│ Begin Again: Special           ┆ Edit TranslationEnglishEspañol… 

#### 8. Genres and Tags exploration


In [16]:
# list columns — need explode() to analyze them
print("--- Top Genres ---")
print(
    df.explode("genres")
    .group_by("genres")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

print("\n--- Top Tags ---")
print(
    df.explode("tags")
    .group_by("tags")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(20)
)

# Check for empty genres or tags lists
empty_genres = df.filter(pl.col("genres").list.len() == 0)
empty_tags = df.filter(pl.col("tags").list.len() == 0)

print(f"Dramas with empty genres list: {empty_genres.height}")
print(f"Dramas with empty tags list: {empty_tags.height}")

--- Top Genres ---
shape: (15, 2)
┌────────────┬───────┐
│ genres     ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ Romance    ┆ 1557  │
│ Drama      ┆ 1064  │
│ Historical ┆ 777   │
│ Comedy     ┆ 631   │
│ Mystery    ┆ 448   │
│ …          ┆ …     │
│ Action     ┆ 214   │
│ Family     ┆ 167   │
│ War        ┆ 142   │
│ Crime      ┆ 137   │
│ Business   ┆ 137   │
└────────────┴───────┘

--- Top Tags ---
shape: (20, 2)
┌──────────────────────────┬───────┐
│ tags                     ┆ count │
│ ---                      ┆ ---   │
│ str                      ┆ u32   │
╞══════════════════════════╪═══════╡
│ Web Series               ┆ 477   │
│ Historical Fiction       ┆ 374   │
│ Adapted From A Novel     ┆ 369   │
│ Short Length Series      ┆ 348   │
│ Investigation            ┆ 318   │
│ …                        ┆ …     │
│ Adventure                ┆ 148   │
│ Smart Female Lead        ┆ 147   │
│ Action                   ┆ 146   │
│ Adapted From A Web Nov

#### 9. Year distribution

In [15]:
# How old is the data? Any year gaps?
print(
    df.group_by("year")
    .agg(pl.len().alias("count"))
    .sort("year")
)

shape: (39, 2)
┌──────┬───────┐
│ year ┆ count │
│ ---  ┆ ---   │
│ i64  ┆ u32   │
╞══════╪═══════╡
│ null ┆ 82    │
│ 1987 ┆ 2     │
│ 1988 ┆ 1     │
│ 1990 ┆ 2     │
│ 1991 ┆ 1     │
│ …    ┆ …     │
│ 2022 ┆ 239   │
│ 2023 ┆ 240   │
│ 2024 ┆ 203   │
│ 2025 ┆ 305   │
│ 2026 ┆ 39    │
└──────┴───────┘


#### 10. Summary
Total dramas - 2,694
- schema - all types correctly inferred by Polars
- no duplicate mld_ids, all rows unique
- scores
  - 11 dramas below 7.5 slipped through scraper filter
  - 192 dramas rated above 9.0, 70 of those are > 10.0 => confirmed scraper bug;
- watchers - 1,176 dramas with < 500 watchers - likely too obscure to give reliable recommendations
- episodes - 132 dramas with < 10 episodes (specials, extras, mini-series)
- synopsis
  - 0 null or empty synopses
  - All synopses contain trailing junk:
   '(Source: MyDramaList)...Edit TranslationEnglish한국어...'
   Shortest synopses (~50 chars) are likely JUST this junk text
   → after stripping, they'll effectively be empty
- genres and tags - 12 dramas with empty genres list and 258 dramas with empty tags list (expected to improve after cleaning)